In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score

import matplotlib.pyplot as plt

In [2]:
# Change this path if your CSV is in another folder
DATA_PATH = "flights_weather_sample.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (9197, 28)


,flight_id,date,day_of_week,carrier,flight_number,tail_number,origin,origin_lat,origin_lon,dest,...,snowfall_cm,cloud_cover_pct,weather_code,dep_delay_min,arr_delay_min,delayed_15,cancelled,delay_cause,late_aircraft_delay_min,weather_delay_min
0,2015-01-05_DL2582,2015-01-05,1,DL,2582,N907DA,DEN,39.85841,-104.66700,ATL,...,0.0,1,0,4.0,-19.0,0.0,0,none,NaN,NaN
1,2015-01-05_US2055,2015-01-05,1,US,2055,N183UW,DEN,39.85841,-104.66700,PHL,...,0.0,1,0,24.0,31.0,1.0,0,carrier,0.0,0.0
2,2015-01-05_US1725,2015-01-05,1,US,1725,N576UW,DEN,39.85841,-104.66700,CLT,...,0.0,0,0,-4.0,18.0,0.0,0,nas,0.0,0.0
3,2015-01-05_AA2392,2015-01-05,1,AA,2392,N3KFAA,DEN,39.85841,-104.66700,MIA,...,0.0,0,0,9.0,-3.0,0.0,0,none,NaN,NaN
4,2015-01-05_US602,2015-01-05,1,US,602,N545UW,ORD,41.97960,-87.90446,PHX,...,0.0,7,0,16.0,20.0,1.0,0,weather,0.0,16.0


In [3]:
print(df.columns.tolist())
print(df.info())

['flight_id', 'date', 'day_of_week', 'carrier', 'flight_number', 'tail_number', 'origin', 'origin_lat', 'origin_lon', 'dest', 'sched_dep_local', 'dep_hour', 'scheduled_arr_local', 'distance_km', 'temp_c', 'wind_speed_kmh', 'wind_gust_kmh', 'precip_mm', 'snowfall_cm', 'cloud_cover_pct', 'weather_code', 'dep_delay_min', 'arr_delay_min', 'delayed_15', 'cancelled', 'delay_cause', 'late_aircraft_delay_min', 'weather_delay_min']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9197 entries, 0 to 9196
Data columns (total 28 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   flight_id                9197 non-null   object 
 1   date                     9197 non-null   object 
 2   day_of_week              9197 non-null   int64  
 3   carrier                  9197 non-null   object 
 4   flight_number            9197 non-null   int64  
 5   tail_number              9154 non-null   object 
 6   origin                   9197

In [4]:
# Combine date and scheduled departure time
df["sched_dep_dt"] = pd.to_datetime(
    df["date"].astype(str) + " " + df["sched_dep_local"].astype(str),
    errors="coerce"
)

# Combine date and scheduled arrival time
df["scheduled_arr_dt"] = pd.to_datetime(
    df["date"].astype(str) + " " + df["scheduled_arr_local"].astype(str),
    errors="coerce"
)

# If scheduled arrival time is earlier than departure time, assume arrival is next day
overnight_mask = df["scheduled_arr_dt"] < df["sched_dep_dt"]
df.loc[overnight_mask, "scheduled_arr_dt"] += pd.Timedelta(days=1)

df[["date", "sched_dep_local", "scheduled_arr_local", "sched_dep_dt", "scheduled_arr_dt"]].head()

,date,sched_dep_local,scheduled_arr_local,sched_dep_dt,scheduled_arr_dt
0,2015-01-05,00:30,05:29,2015-01-05 00:30:00,2015-01-05 05:29:00
1,2015-01-05,00:45,06:07,2015-01-05 00:45:00,2015-01-05 06:07:00
2,2015-01-05,01:20,06:18,2015-01-05 01:20:00,2015-01-05 06:18:00
3,2015-01-05,01:20,07:07,2015-01-05 01:20:00,2015-01-05 07:07:00
4,2015-01-05,05:00,07:48,2015-01-05 05:00:00,2015-01-05 07:48:00


In [5]:
df = df.sort_values("sched_dep_dt").reset_index(drop=True)

df[["flight_id", "tail_number", "origin", "dest", "sched_dep_dt", "scheduled_arr_dt"]].head()

,flight_id,tail_number,origin,dest,sched_dep_dt,scheduled_arr_dt
0,2015-01-05_DL2582,N907DA,DEN,ATL,2015-01-05 00:30:00,2015-01-05 05:29:00
1,2015-01-05_US2055,N183UW,DEN,PHL,2015-01-05 00:45:00,2015-01-05 06:07:00
2,2015-01-05_US1725,N576UW,DEN,CLT,2015-01-05 01:20:00,2015-01-05 06:18:00
3,2015-01-05_AA2392,N3KFAA,DEN,MIA,2015-01-05 01:20:00,2015-01-05 07:07:00
4,2015-01-05_US602,N545UW,ORD,PHX,2015-01-05 05:00:00,2015-01-05 07:48:00


In [6]:
# Sort by aircraft and scheduled departure time
df = df.sort_values(["tail_number", "sched_dep_dt"]).reset_index(drop=True)

# Previous flight details for the same physical aircraft
df["prev_flight_id"] = df.groupby("tail_number")["flight_id"].shift(1)
df["prev_origin"] = df.groupby("tail_number")["origin"].shift(1)
df["prev_dest"] = df.groupby("tail_number")["dest"].shift(1)
df["prev_sched_dep_dt"] = df.groupby("tail_number")["sched_dep_dt"].shift(1)
df["prev_scheduled_arr_dt"] = df.groupby("tail_number")["scheduled_arr_dt"].shift(1)

# Previous flight outcome, used only because it happened before current flight
df["prev_arr_delay_min"] = df.groupby("tail_number")["arr_delay_min"].shift(1)
df["prev_dep_delay_min"] = df.groupby("tail_number")["dep_delay_min"].shift(1)

# Check whether previous flight destination equals current origin
# This makes the cascade signal more physically meaningful
df["prev_dest_equals_current_origin"] = (df["prev_dest"] == df["origin"]).astype(int)

# Approximate turnaround buffer in minutes
df["turnaround_buffer_min"] = (
    (df["sched_dep_dt"] - df["prev_scheduled_arr_dt"]).dt.total_seconds() / 60
)

# If previous destination is not current origin, turnaround buffer is unreliable
df.loc[df["prev_dest_equals_current_origin"] == 0, "turnaround_buffer_min"] = np.nan

# Fill missing previous delay with 0 for model input
df["prev_arr_delay_min_filled"] = df["prev_arr_delay_min"].fillna(0)
df["prev_dep_delay_min_filled"] = df["prev_dep_delay_min"].fillna(0)

# Cascade risk flags
df["prev_arr_delayed_15"] = (df["prev_arr_delay_min_filled"] >= 15).astype(int)
df["prev_arr_heavily_delayed_30"] = (df["prev_arr_delay_min_filled"] >= 30).astype(int)

# Recovery gap:
# positive = schedule has extra buffer
# negative = previous delay is larger than available turnaround buffer
df["recovery_gap_min"] = df["turnaround_buffer_min"] - df["prev_arr_delay_min_filled"]

df[[
    "flight_id", "tail_number", "origin", "dest",
    "prev_flight_id", "prev_dest", "prev_dest_equals_current_origin",
    "prev_arr_delay_min", "turnaround_buffer_min", "recovery_gap_min"
]].head(20)

,flight_id,tail_number,origin,dest,prev_flight_id,prev_dest,prev_dest_equals_current_origin,prev_arr_delay_min,turnaround_buffer_min,recovery_gap_min
0,2015-01-07_AA1080,N004AA,ORD,EGE,NaN,NaN,0,NaN,NaN,NaN
1,2015-01-07_AA2349,N004AA,ORD,DFW,2015-01-07_AA1080,EGE,0,91.0,NaN,NaN
2,2015-01-10_AA1344,N009AA,DEN,DFW,NaN,NaN,0,NaN,NaN,NaN
3,2015-01-11_AA1080,N012AA,ORD,EGE,NaN,NaN,0,NaN,NaN,NaN
4,2015-01-11_AA2349,N012AA,ORD,DFW,2015-01-11_AA1080,EGE,0,-13.0,NaN,NaN
5,2015-01-10_AA1080,N013AA,ORD,EGE,NaN,NaN,0,NaN,NaN,NaN
6,2015-01-10_AA2349,N013AA,ORD,DFW,2015-01-10_AA1080,EGE,0,9.0,NaN,NaN
7,2015-01-05_AA1080,N015AA,ORD,EGE,NaN,NaN,0,NaN,NaN,NaN
8,2015-01-08_AA1080,N015AA,ORD,EGE,2015-01-05_AA1080,EGE,0,NaN,NaN,NaN
9,2015-01-08_AA2349,N015AA,ORD,DFW,2015-01-08_AA1080,EGE,0,53.0,NaN,NaN


In [7]:
# Weather flags
df["snow_flag"] = (df["snowfall_cm"] > 0).astype(int)
df["rain_or_precip_flag"] = (df["precip_mm"] > 0).astype(int)
df["high_wind_flag"] = (df["wind_gust_kmh"] >= 40).astype(int)

# Weather code groups based on WMO categories
df["fog_flag"] = df["weather_code"].isin([45, 48]).astype(int)
df["snow_weather_code_flag"] = df["weather_code"].between(71, 77).astype(int)
df["rain_weather_code_flag"] = df["weather_code"].between(51, 67).astype(int)
df["storm_flag"] = (df["weather_code"] == 95).astype(int)

# Simple weather severity score
df["weather_severity_score"] = (
    15 * df["snow_flag"]
    + 10 * df["rain_or_precip_flag"]
    + 10 * df["high_wind_flag"]
    + 10 * df["fog_flag"]
    + 15 * df["snow_weather_code_flag"]
    + 10 * df["rain_weather_code_flag"]
    + 20 * df["storm_flag"]
)

# Congestion-prone bank hours
df["morning_bank_flag"] = df["dep_hour"].isin([6, 7, 8, 9]).astype(int)
df["evening_bank_flag"] = df["dep_hour"].isin([16, 17, 18, 19, 20]).astype(int)

# Number of flights scheduled from same origin in same hour
df["origin_hourly_flight_count"] = (
    df.groupby(["origin", "date", "dep_hour"])["flight_id"]
    .transform("count")
)

df[[
    "flight_id", "origin", "dep_hour",
    "snowfall_cm", "precip_mm", "wind_gust_kmh",
    "weather_severity_score",
    "origin_hourly_flight_count"
]].head()

,flight_id,origin,dep_hour,snowfall_cm,precip_mm,wind_gust_kmh,weather_severity_score,origin_hourly_flight_count
0,2015-01-07_AA1080,ORD,11,0.00,0.0,53.6,10,33
1,2015-01-07_AA2349,ORD,18,0.00,0.0,35.3,0,60
2,2015-01-10_AA1344,DEN,17,0.00,0.0,14.4,0,25
3,2015-01-11_AA1080,ORD,11,0.00,0.0,24.5,0,38
4,2015-01-11_AA2349,ORD,18,0.07,0.0,17.3,30,65


In [8]:
# Keep only non-cancelled rows with valid delayed_15 label
model_df = df[
    (df["cancelled"] == 0) &
    (df["delayed_15"].notna())
].copy()

model_df["delayed_15"] = model_df["delayed_15"].astype(int)

print("Model dataset shape:", model_df.shape)
print(model_df["delayed_15"].value_counts(normalize=True))

Model dataset shape: (8514, 55)
delayed_15
1    0.515386
0    0.484614
Name: proportion, dtype: float64


In [9]:
target = "delayed_15"

numeric_features = [
    "day_of_week",
    "dep_hour",
    "distance_km",
    "temp_c",
    "wind_speed_kmh",
    "wind_gust_kmh",
    "precip_mm",
    "snowfall_cm",
    "cloud_cover_pct",
    "weather_code",
    "prev_arr_delay_min_filled",
    "prev_dep_delay_min_filled",
    "prev_arr_delayed_15",
    "prev_arr_heavily_delayed_30",
    "prev_dest_equals_current_origin",
    "turnaround_buffer_min",
    "recovery_gap_min",
    "weather_severity_score",
    "snow_flag",
    "rain_or_precip_flag",
    "high_wind_flag",
    "fog_flag",
    "snow_weather_code_flag",
    "rain_weather_code_flag",
    "storm_flag",
    "morning_bank_flag",
    "evening_bank_flag",
    "origin_hourly_flight_count"
]

categorical_features = [
    "carrier",
    "origin",
    "dest"
]

features = numeric_features + categorical_features

X = model_df[features].copy()
y = model_df[target].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (8514, 31)
y shape: (8514,)


In [10]:
# Sort chronologically before splitting
model_df = model_df.sort_values("sched_dep_dt").reset_index(drop=True)

X = model_df[features].copy()
y = model_df[target].copy()

test_size = 0.15
split_index = int(len(model_df) * (1 - test_size))

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

train_meta = model_df.iloc[:split_index].copy()
test_meta = model_df.iloc[split_index:].copy()

print("Total rows:", len(model_df))
print("Train rows:", len(X_train))
print("Test rows:", len(X_test))

print("Train date range:")
print(train_meta["sched_dep_dt"].min(), "to", train_meta["sched_dep_dt"].max())

print("Test date range:")
print(test_meta["sched_dep_dt"].min(), "to", test_meta["sched_dep_dt"].max())

Total rows: 8514
Train rows: 7236
Test rows: 1278
Train date range:
2015-01-05 00:30:00 to 2015-01-10 19:15:00
Test date range:
2015-01-10 19:15:00 to 2015-01-11 23:59:00


In [11]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

log_reg_model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    solver="liblinear",
    random_state=42
)

clf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", log_reg_model)
])

clf

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['day_of_week', 'dep_hour',
                                                   'distance_km', 'temp_c',
                                                   'wind_speed_kmh',
                                                   'wind_gust_kmh', 'precip_mm',
                                                   'snowfall_cm',
                                                   'cloud_cover_pct',
                                                   'weather_code',
                                                   'prev_arr_delay_min_filled',
                                                   'prev_dep_delay_min...
                                                   'rain_weather_code_flag',
                                                   'storm_flag',
                                                   'morning_bank_flag',
                                                   'evening_bank_flag',
                                                   'origin_hourly_flight_count']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['carrier', 'origin',
                                                   'dest'])])),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=2000,
                                    random_state=42, solver='liblinear'))])

In [12]:
clf.fit(X_train, y_train)

print("Model training completed.")

Model training completed.


In [13]:
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.6807511737089202
ROC AUC: 0.6836298913504796

Confusion Matrix:
[[694 122]
 [286 176]]

Classification Report:
              precision    recall  f1-score   support

           0       0.71      0.85      0.77       816
           1       0.59      0.38      0.46       462

    accuracy                           0.68      1278
   macro avg       0.65      0.62      0.62      1278
weighted avg       0.67      0.68      0.66      1278



In [14]:
# Baseline: always predict not delayed
baseline_pred = np.zeros_like(y_test)

print("Baseline Accuracy:", accuracy_score(y_test, baseline_pred))
print("\nBaseline Classification Report:")
print(classification_report(y_test, baseline_pred))

Baseline Accuracy: 0.6384976525821596

Baseline Classification Report:
              precision    recall  f1-score   support

           0       0.64      1.00      0.78       816
           1       0.00      0.00      0.00       462

    accuracy                           0.64      1278
   macro avg       0.32      0.50      0.39      1278
weighted avg       0.41      0.64      0.50      1278



/Users/bhargavreddy/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/bhargavreddy/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/bhargavreddy/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitaliz

In [15]:
test_results = test_meta.copy()

test_results["predicted_delay_probability"] = y_prob
test_results["predicted_delayed_15"] = y_pred

def risk_level(prob):
    if prob >= 0.65:
        return "High"
    elif prob >= 0.35:
        return "Medium"
    else:
        return "Low"

test_results["risk_level"] = test_results["predicted_delay_probability"].apply(risk_level)

test_results[[
    "flight_id", "sched_dep_dt", "carrier", "origin", "dest",
    "predicted_delay_probability", "risk_level",
    "delayed_15", "delay_cause"
]].head(20)

,flight_id,sched_dep_dt,carrier,origin,dest,predicted_delay_probability,risk_level,delayed_15,delay_cause
7236,2015-01-10_MQ3280,2015-01-10 19:15:00,MQ,ORD,RST,0.911893,High,1,weather
7237,2015-01-10_F9626,2015-01-10 19:15:00,F9,DEN,DTW,0.281078,Low,1,none
7238,2015-01-10_AA1617,2015-01-10 19:15:00,AA,ORD,LAX,0.404000,Medium,0,none
7239,2015-01-10_AS21,2015-01-10 19:30:00,AS,ORD,SEA,0.338621,Low,0,nas
7240,2015-01-10_WN663,2015-01-10 19:30:00,WN,DEN,LAS,0.360182,Medium,0,none
7241,2015-01-10_OO6277,2015-01-10 19:35:00,OO,ORD,DCA,0.525371,Medium,1,weather
7242,2015-01-10_EV4755,2015-01-10 19:35:00,EV,ORD,CAE,0.371765,Medium,1,late_aircraft
7243,2015-01-10_UA1074,2015-01-10 19:35:00,UA,ORD,BWI,0.440370,Medium,0,none
7244,2015-01-10_UA1650,2015-01-10 19:35:00,UA,ORD,ATL,0.525185,Medium,1,weather
7245,2015-01-10_EV4560,2015-01-10 19:35:00,EV,ORD,CLT,0.333847,Low,0,none


In [16]:
def explain_flight(row):
    reasons = []
    
    # Cascade reason
    if row["prev_arr_delay_min_filled"] >= 30:
        reasons.append(
            f"Strong cascade risk: previous aircraft arrival delay was {row['prev_arr_delay_min_filled']:.0f} min."
        )
    elif row["prev_arr_delay_min_filled"] >= 15:
        reasons.append(
            f"Moderate cascade risk: previous aircraft arrival delay was {row['prev_arr_delay_min_filled']:.0f} min."
        )
    
    # Turnaround reason
    if pd.notna(row["turnaround_buffer_min"]):
        if row["turnaround_buffer_min"] < 30:
            reasons.append(
                f"Short turnaround buffer: only {row['turnaround_buffer_min']:.0f} min available."
            )
        if pd.notna(row["recovery_gap_min"]) and row["recovery_gap_min"] < 0:
            reasons.append(
                f"Negative recovery gap: previous delay exceeds turnaround buffer by {abs(row['recovery_gap_min']):.0f} min."
            )
    
    # Weather reason
    if row["snowfall_cm"] > 0:
        reasons.append(f"Snow at origin: {row['snowfall_cm']:.2f} cm during departure hour.")
    
    if row["precip_mm"] > 0:
        reasons.append(f"Precipitation at origin: {row['precip_mm']:.2f} mm.")
    
    if row["wind_gust_kmh"] >= 40:
        reasons.append(f"High wind gusts: {row['wind_gust_kmh']:.0f} km/h.")
    
    if row["fog_flag"] == 1:
        reasons.append("Fog condition indicated by weather code.")
    
    # Congestion reason
    if row["evening_bank_flag"] == 1:
        reasons.append("Departure is during evening congestion-prone bank.")
    
    if row["morning_bank_flag"] == 1:
        reasons.append("Departure is during morning congestion-prone bank.")
    
    if row["origin_hourly_flight_count"] >= 40:
        reasons.append(
            f"High airport departure load: {row['origin_hourly_flight_count']} flights scheduled from {row['origin']} in this hour."
        )
    
    if len(reasons) == 0:
        reasons.append("No single strong driver found; risk may come from weaker combined schedule/weather factors.")
    
    return reasons


def main_reason_category(row):
    if row["prev_arr_delay_min_filled"] >= 15:
        return "Cascade / late inbound aircraft"
    
    if (
        row["snowfall_cm"] > 0 or
        row["precip_mm"] > 0 or
        row["wind_gust_kmh"] >= 40 or
        row["fog_flag"] == 1
    ):
        return "Weather pressure"
    
    if (
        row["morning_bank_flag"] == 1 or
        row["evening_bank_flag"] == 1 or
        row["origin_hourly_flight_count"] >= 40
    ):
        return "Congestion / schedule pressure"
    
    return "General operational risk"


def recommend_action(row):
    reason = row["main_reason"]
    
    if reason == "Cascade / late inbound aircraft":
        return "Monitor inbound tail, alert gate team, prepare fast-turnaround support, and check backup aircraft availability."
    
    elif reason == "Weather pressure":
        return "Alert ground operations, monitor airport-wide weather impact, and prepare extra turnaround buffer."
    
    elif reason == "Congestion / schedule pressure":
        return "Prioritize high-connection flights and coordinate pushback or gate sequencing with operations control."
    
    else:
        return "Keep under monitoring; no immediate strong intervention trigger found."

In [17]:
test_results["main_reason"] = test_results.apply(main_reason_category, axis=1)
test_results["recommended_action"] = test_results.apply(recommend_action, axis=1)
test_results["explanation_list"] = test_results.apply(explain_flight, axis=1)
test_results["explanation"] = test_results["explanation_list"].apply(lambda x: " | ".join(x))

test_results[[
    "flight_id", "sched_dep_dt", "origin", "dest",
    "predicted_delay_probability", "risk_level",
    "main_reason", "explanation", "recommended_action",
    "delayed_15", "delay_cause"
]].head(20)

,flight_id,sched_dep_dt,origin,dest,predicted_delay_probability,risk_level,main_reason,explanation,recommended_action,delayed_15,delay_cause
7236,2015-01-10_MQ3280,2015-01-10 19:15:00,ORD,RST,0.911893,High,Cascade / late inbound aircraft,Moderate cascade risk: previous aircraft arriv...,"Monitor inbound tail, alert gate team, prepare...",1,weather
7237,2015-01-10_F9626,2015-01-10 19:15:00,DEN,DTW,0.281078,Low,Cascade / late inbound aircraft,Moderate cascade risk: previous aircraft arriv...,"Monitor inbound tail, alert gate team, prepare...",1,none
7238,2015-01-10_AA1617,2015-01-10 19:15:00,ORD,LAX,0.404000,Medium,Congestion / schedule pressure,Departure is during evening congestion-prone b...,Prioritize high-connection flights and coordin...,0,none
7239,2015-01-10_AS21,2015-01-10 19:30:00,ORD,SEA,0.338621,Low,Congestion / schedule pressure,Departure is during evening congestion-prone b...,Prioritize high-connection flights and coordin...,0,nas
7240,2015-01-10_WN663,2015-01-10 19:30:00,DEN,LAS,0.360182,Medium,Cascade / late inbound aircraft,Moderate cascade risk: previous aircraft arriv...,"Monitor inbound tail, alert gate team, prepare...",0,none
7241,2015-01-10_OO6277,2015-01-10 19:35:00,ORD,DCA,0.525371,Medium,Congestion / schedule pressure,Departure is during evening congestion-prone b...,Prioritize high-connection flights and coordin...,1,weather
7242,2015-01-10_EV4755,2015-01-10 19:35:00,ORD,CAE,0.371765,Medium,Cascade / late inbound aircraft,Strong cascade risk: previous aircraft arrival...,"Monitor inbound tail, alert gate team, prepare...",1,late_aircraft
7243,2015-01-10_UA1074,2015-01-10 19:35:00,ORD,BWI,0.440370,Medium,Congestion / schedule pressure,Departure is during evening congestion-prone b...,Prioritize high-connection flights and coordin...,0,none
7244,2015-01-10_UA1650,2015-01-10 19:35:00,ORD,ATL,0.525185,Medium,Congestion / schedule pressure,Departure is during evening congestion-prone b...,Prioritize high-connection flights and coordin...,1,weather
7245,2015-01-10_EV4560,2015-01-10 19:35:00,ORD,CLT,0.333847,Low,Congestion / schedule pressure,Departure is during evening congestion-prone b...,Prioritize high-connection flights and coordin...,0,none


In [18]:
high_risk = test_results[test_results["risk_level"] == "High"].copy()

high_risk = high_risk.sort_values(
    "predicted_delay_probability",
    ascending=False
)

high_risk[[
    "flight_id", "sched_dep_dt", "carrier", "origin", "dest",
    "predicted_delay_probability", "risk_level",
    "main_reason",
    "explanation",
    "recommended_action",
    "delayed_15",
    "delay_cause"
]].head(20)

,flight_id,sched_dep_dt,carrier,origin,dest,predicted_delay_probability,risk_level,main_reason,explanation,recommended_action,delayed_15,delay_cause
8421,2015-01-11_MQ2975,2015-01-11 20:20:00,MQ,ORD,HSV,0.989106,High,Cascade / late inbound aircraft,Strong cascade risk: previous aircraft arrival...,"Monitor inbound tail, alert gate team, prepare...",1,carrier
8399,2015-01-11_MQ3280,2015-01-11 20:05:00,MQ,ORD,RST,0.988099,High,Weather pressure,Snow at origin: 0.28 cm during departure hour....,"Alert ground operations, monitor airport-wide ...",1,nas
8490,2015-01-11_MQ3174,2015-01-11 21:55:00,MQ,ORD,LIT,0.984766,High,Cascade / late inbound aircraft,Strong cascade risk: previous aircraft arrival...,"Monitor inbound tail, alert gate team, prepare...",1,weather
8390,2015-01-11_MQ3185,2015-01-11 19:55:00,MQ,ORD,COU,0.982028,High,Cascade / late inbound aircraft,Strong cascade risk: previous aircraft arrival...,"Monitor inbound tail, alert gate team, prepare...",1,nas
8409,2015-01-11_MQ3039,2015-01-11 20:10:00,MQ,ORD,ALO,0.980839,High,Weather pressure,Snow at origin: 0.28 cm during departure hour....,"Alert ground operations, monitor airport-wide ...",1,weather
8408,2015-01-11_OO2644,2015-01-11 20:10:00,OO,ORD,CHO,0.978686,High,Cascade / late inbound aircraft,Strong cascade risk: previous aircraft arrival...,"Monitor inbound tail, alert gate team, prepare...",0,nas
8432,2015-01-11_MQ3673,2015-01-11 20:35:00,MQ,ORD,CMI,0.976985,High,Weather pressure,Snow at origin: 0.28 cm during departure hour....,"Alert ground operations, monitor airport-wide ...",1,nas
8461,2015-01-11_MQ3107,2015-01-11 21:10:00,MQ,ORD,GRR,0.975956,High,Cascade / late inbound aircraft,Strong cascade risk: previous aircraft arrival...,"Monitor inbound tail, alert gate team, prepare...",1,nas
8326,2015-01-11_MQ3676,2015-01-11 19:10:00,MQ,ORD,LEX,0.973446,High,Weather pressure,Snow at origin: 0.28 cm during departure hour....,"Alert ground operations, monitor airport-wide ...",0,nas
8439,2015-01-11_MQ3608,2015-01-11 20:55:00,MQ,ORD,GRB,0.972533,High,Cascade / late inbound aircraft,Strong cascade risk: previous aircraft arrival...,"Monitor inbound tail, alert gate team, prepare...",1,nas


In [19]:
# Change this index to inspect another high-risk flight
selected_index = 0

if len(high_risk) > 0:
    row = high_risk.iloc[selected_index]
    
    print("Flight:", row["flight_id"])
    print("Route:", row["origin"], "to", row["dest"])
    print("Scheduled departure:", row["sched_dep_dt"])
    print("Predicted delay probability:", round(row["predicted_delay_probability"], 3))
    print("Risk level:", row["risk_level"])
    print("Main reason:", row["main_reason"])
    print("\nExplanation:")
    for reason in row["explanation_list"]:
        print("-", reason)
    print("\nRecommended action:")
    print(row["recommended_action"])
    print("\nActual delayed_15:", row["delayed_15"])
    print("Actual delay cause:", row["delay_cause"])
else:
    print("No high-risk flights found in the test set.")

Flight: 2015-01-11_MQ2975
Route: ORD to HSV
Scheduled departure: 2015-01-11 20:20:00
Predicted delay probability: 0.989
Risk level: High
Main reason: Cascade / late inbound aircraft

Explanation:
- Strong cascade risk: previous aircraft arrival delay was 84 min.
- Snow at origin: 0.28 cm during departure hour.
- Precipitation at origin: 0.40 mm.
- Departure is during evening congestion-prone bank.
- High airport departure load: 52 flights scheduled from ORD in this hour.

Recommended action:
Monitor inbound tail, alert gate team, prepare fast-turnaround support, and check backup aircraft availability.

Actual delayed_15: 1
Actual delay cause: carrier


In [20]:
# Get feature names after preprocessing
preprocessor_fitted = clf.named_steps["preprocessor"]
model_fitted = clf.named_steps["model"]

feature_names = preprocessor_fitted.get_feature_names_out()
coefficients = model_fitted.coef_[0]

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
})

coef_df["abs_coefficient"] = coef_df["coefficient"].abs()

coef_df_sorted = coef_df.sort_values("abs_coefficient", ascending=False)

coef_df_sorted.head(30)

,feature,coefficient,abs_coefficient
34,cat__carrier_MQ,1.756092,1.756092
114,cat__dest_GCC,-1.409768,1.409768
48,cat__dest_ANC,1.233183,1.233183
73,cat__dest_CHO,1.101296,1.101296
80,cat__dest_CMX,1.048104,1.048104
96,cat__dest_DRO,-0.942959,0.942959
205,cat__dest_RST,0.934994,0.934994
39,cat__carrier_VX,-0.905616,0.905616
109,cat__dest_FCA,0.901078,0.901078
220,cat__dest_SJC,-0.863525,0.863525


In [21]:
output_cols = [
    "flight_id",
    "sched_dep_dt",
    "carrier",
    "flight_number",
    "tail_number",
    "origin",
    "dest",
    "dep_hour",
    "predicted_delay_probability",
    "predicted_delayed_15",
    "risk_level",
    "main_reason",
    "explanation",
    "recommended_action",
    "delayed_15",
    "dep_delay_min",
    "arr_delay_min",
    "delay_cause",
    "late_aircraft_delay_min",
    "weather_delay_min"
]

available_output_cols = [col for col in output_cols if col in test_results.columns]

test_results[available_output_cols].to_csv(
    "explainable_delay_risk_test_results.csv",
    index=False
)

print("Saved: explainable_delay_risk_test_results.csv")

Saved: explainable_delay_risk_test_results.csv


In [22]:
display_cols = [
    "flight_id",
    "sched_dep_dt",
    "origin",
    "dest",
    "predicted_delay_probability",
    "risk_level",
    "main_reason",
    "recommended_action",
    "delayed_15",
    "delay_cause"
]

test_results[display_cols].sort_values(
    "predicted_delay_probability",
    ascending=False
).head(30)

,flight_id,sched_dep_dt,origin,dest,predicted_delay_probability,risk_level,main_reason,recommended_action,delayed_15,delay_cause
8421,2015-01-11_MQ2975,2015-01-11 20:20:00,ORD,HSV,0.989106,High,Cascade / late inbound aircraft,"Monitor inbound tail, alert gate team, prepare...",1,carrier
8399,2015-01-11_MQ3280,2015-01-11 20:05:00,ORD,RST,0.988099,High,Weather pressure,"Alert ground operations, monitor airport-wide ...",1,nas
8490,2015-01-11_MQ3174,2015-01-11 21:55:00,ORD,LIT,0.984766,High,Cascade / late inbound aircraft,"Monitor inbound tail, alert gate team, prepare...",1,weather
8390,2015-01-11_MQ3185,2015-01-11 19:55:00,ORD,COU,0.982028,High,Cascade / late inbound aircraft,"Monitor inbound tail, alert gate team, prepare...",1,nas
8409,2015-01-11_MQ3039,2015-01-11 20:10:00,ORD,ALO,0.980839,High,Weather pressure,"Alert ground operations, monitor airport-wide ...",1,weather
8408,2015-01-11_OO2644,2015-01-11 20:10:00,ORD,CHO,0.978686,High,Cascade / late inbound aircraft,"Monitor inbound tail, alert gate team, prepare...",0,nas
8432,2015-01-11_MQ3673,2015-01-11 20:35:00,ORD,CMI,0.976985,High,Weather pressure,"Alert ground operations, monitor airport-wide ...",1,nas
8461,2015-01-11_MQ3107,2015-01-11 21:10:00,ORD,GRR,0.975956,High,Cascade / late inbound aircraft,"Monitor inbound tail, alert gate team, prepare...",1,nas
8326,2015-01-11_MQ3676,2015-01-11 19:10:00,ORD,LEX,0.973446,High,Weather pressure,"Alert ground operations, monitor airport-wide ...",0,nas
8439,2015-01-11_MQ3608,2015-01-11 20:55:00,ORD,GRB,0.972533,High,Cascade / late inbound aircraft,"Monitor inbound tail, alert gate team, prepare...",1,nas


In [23]:
import joblib
import json
from pathlib import Path

# Create folder for saved model
MODEL_DIR = Path("saved_model")
MODEL_DIR.mkdir(exist_ok=True)

# Save the full pipeline: preprocessing + logistic regression
joblib.dump(clf, MODEL_DIR / "delay_risk_logistic_regression_pipeline.joblib")

# Save feature information separately
model_info = {
    "target": target,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "features": features,
    "model_type": "Logistic Regression",
    "test_split": "Last 15% chronological split",
    "risk_thresholds": {
        "High": "probability >= 0.65",
        "Medium": "0.35 <= probability < 0.65",
        "Low": "probability < 0.35"
    }
}

with open(MODEL_DIR / "model_info.json", "w") as f:
    json.dump(model_info, f, indent=4)

print("Model saved successfully.")
print("Saved files:")
print(MODEL_DIR / "delay_risk_logistic_regression_pipeline.joblib")
print(MODEL_DIR / "model_info.json")

Model saved successfully.
Saved files:
saved_model/delay_risk_logistic_regression_pipeline.joblib
saved_model/model_info.json


In [24]:
#load saved model
import joblib
import json

# Load trained pipeline
loaded_model = joblib.load("saved_model/delay_risk_logistic_regression_pipeline.joblib")

# Load model info
with open("saved_model/model_info.json", "r") as f:
    loaded_model_info = json.load(f)

print("Model loaded successfully.")
print(loaded_model_info)

Model loaded successfully.
{'target': 'delayed_15', 'numeric_features': ['day_of_week', 'dep_hour', 'distance_km', 'temp_c', 'wind_speed_kmh', 'wind_gust_kmh', 'precip_mm', 'snowfall_cm', 'cloud_cover_pct', 'weather_code', 'prev_arr_delay_min_filled', 'prev_dep_delay_min_filled', 'prev_arr_delayed_15', 'prev_arr_heavily_delayed_30', 'prev_dest_equals_current_origin', 'turnaround_buffer_min', 'recovery_gap_min', 'weather_severity_score', 'snow_flag', 'rain_or_precip_flag', 'high_wind_flag', 'fog_flag', 'snow_weather_code_flag', 'rain_weather_code_flag', 'storm_flag', 'morning_bank_flag', 'evening_bank_flag', 'origin_hourly_flight_count'], 'categorical_features': ['carrier', 'origin', 'dest'], 'features': ['day_of_week', 'dep_hour', 'distance_km', 'temp_c', 'wind_speed_kmh', 'wind_gust_kmh', 'precip_mm', 'snowfall_cm', 'cloud_cover_pct', 'weather_code', 'prev_arr_delay_min_filled', 'prev_dep_delay_min_filled', 'prev_arr_delayed_15', 'prev_arr_heavily_delayed_30', 'prev_dest_equals_curren

In [25]:
# Example: predict again on test data
loaded_prob = loaded_model.predict_proba(X_test)[:, 1]
loaded_pred = loaded_model.predict(X_test)

test_results_loaded = test_meta.copy()
test_results_loaded["predicted_delay_probability"] = loaded_prob
test_results_loaded["predicted_delayed_15"] = loaded_pred

test_results_loaded[[
    "flight_id",
    "sched_dep_dt",
    "origin",
    "dest",
    "predicted_delay_probability",
    "predicted_delayed_15",
    "delayed_15"
]].head()

,flight_id,sched_dep_dt,origin,dest,predicted_delay_probability,predicted_delayed_15,delayed_15
7236,2015-01-10_MQ3280,2015-01-10 19:15:00,ORD,RST,0.911893,1,1
7237,2015-01-10_F9626,2015-01-10 19:15:00,DEN,DTW,0.281078,0,1
7238,2015-01-10_AA1617,2015-01-10 19:15:00,ORD,LAX,0.404000,0,0
7239,2015-01-10_AS21,2015-01-10 19:30:00,ORD,SEA,0.338621,0,0
7240,2015-01-10_WN663,2015-01-10 19:30:00,DEN,LAS,0.360182,0,0
